In [0]:
from pyspark.sql import functions as F

# -----------------------------
# 0) Parâmetros do ambiente
# -----------------------------
CATALOG = "workspace"
SCHEMA_FILES = "default"
VOLUME = "landing_dados"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA_FILES}/{VOLUME}"

FILES = {
    "brasileirao": f"{BASE_PATH}/brasileirao.csv",
    "times":       f"{BASE_PATH}/times.csv",
    "estadio":     f"{BASE_PATH}/estadios.csv"
}

CSV_SEP = ";"  # você disse que seus arquivos usam ';'


# -----------------------------------
# 1) Garante que o schema landing existe
# -----------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS landing")


# -----------------------------------
# 2) Função: ler CSV e padronizar colunas
# -----------------------------------
def read_csv(path: str):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("sep", CSV_SEP)
        .option("multiLine", True)         # ajuda caso tenha algum texto com quebra de linha
        .option("quote", "\"")             # aspas padrão
        .option("escape", "\"")            # escape padrão
        .csv(path)
    )
    # Padroniza nomes de colunas (remove espaços nas pontas)
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip())
    return df


# ----------------------------------------------------
# 3) Função: escrever Delta com overwriteSchema (Opção A)
# ----------------------------------------------------
def write_landing_table(df, table_name: str):
    full_table = f"landing.{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")   # <-- evita metadata mismatch ao recriar schema
        .saveAsTable(full_table)
    )
    return full_table


# -----------------------------
# 4) Ingestão: leitura dos CSVs
# -----------------------------
df_bra = read_csv(FILES["brasileirao"])
df_tim = read_csv(FILES["times"])
df_est = read_csv(FILES["estadio"])

# (Opcional) Mostra amostra e schema para depurar separador/colunas
print("== brasileirao schema ==")
df_bra.printSchema()
display(df_bra.limit(5))

print("== times schema ==")
df_tim.printSchema()
display(df_tim.limit(5))

print("== estadio schema ==")
df_est.printSchema()
display(df_est.limit(5))


# -------------------------------------------------------
# 5) Grava no schema landing como tabelas Delta (Opção A)
# -------------------------------------------------------
t1 = write_landing_table(df_bra, "brasileirao")
t2 = write_landing_table(df_tim, "times")
t3 = write_landing_table(df_est, "estadio")

print("Tabelas criadas/atualizadas com sucesso:")
print(t1, t2, t3)


# -------------------------------------------------------
# 6) Validação final (contagem + colunas)
# -------------------------------------------------------
for t in ["brasileirao", "times", "estadio"]:
    df = spark.table(f"landing.{t}")
    print(f"landing.{t} -> linhas: {df.count()} | colunas: {len(df.columns)}")
